# Block 09: Structured-Noise and Multichannel Robustness

**Goal**  
Stress-test the weighted methods outside the theorem regime without mixing these results into the equivalence claims.

**Checklist alignment**  
Implements E10 and E11, plus a synthetic multichannel characterization note.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
out = run_block09_robustness_suite(cfg)
display(out["nonstationary_df"])
display(out["artifact_df"])
display(out["multichannel_df"])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df = out["nonstationary_df"]
ax.bar(df["case"], df["amplitude_rmse"])
ax.set_ylabel("amplitude RMSE")
ax.set_title("Non-stationary PSD handling")
ax.tick_params(axis="x", rotation=20)
save_figure(fig, dirs["figures"] / "block_09_e10_nonstationary.png")
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
df = out["artifact_df"]
ax.bar(df["pass"], df["amplitude_rmse"])
ax.set_ylabel("amplitude RMSE")
ax.set_title("Artifact flagging and refit")
save_figure(fig, dirs["figures"] / "block_09_e11_artifacts.png")
plt.close(fig)

In [ ]:
ns_path = dirs["tables"] / "block_09_e10_nonstationary.csv"
art_path = dirs["tables"] / "block_09_e11_artifacts.csv"
multi_path = dirs["tables"] / "block_09_multichannel_summary.csv"
manifest_path = dirs["manifests"] / "block_09_robustness_summary.json"
save_dataframe(out["nonstationary_df"], ns_path)
save_dataframe(out["artifact_df"], art_path)
save_dataframe(out["multichannel_df"], multi_path)
save_json({"saved": [str(p.relative_to(REPO_ROOT)) for p in [ns_path, art_path, multi_path]]}, manifest_path)
pd.DataFrame([manifest_row("block_09", "robustness-support", str(ns_path.relative_to(REPO_ROOT)), cfg)])